In [20]:
import numpy as np
import pandas as pd
import polars as pl

import matplotlib.pyplot as plt
from sklearn.tree import plot_tree
from sklearn.model_selection import train_test_split
from sksurv.ensemble import RandomSurvivalForest
from sksurv.linear_model import CoxPHSurvivalAnalysis
from sksurv.metrics import concordance_index_censored , concordance_index_ipcw
from sklearn.impute import SimpleImputer
from sksurv.util import Surv



In [21]:
# Raw data


# Clinical Data
df = pl.read_csv("../data/raw/X_train/clinical_train.csv")
df_eval = pl.read_csv("../data/raw/X_test/clinical_test.csv")

# Molecular Data
mol_df = pl.read_csv("../data/raw/X_train/molecular_train.csv")
mol_eval = pl.read_csv("../data/raw/X_test/molecular_test.csv" , ignore_errors=True)


target_df = pl.read_csv("../data/raw/target_train.csv")

# EDA 

In [22]:
df.head()

ID,CENTER,BM_BLAST,WBC,ANC,MONOCYTES,HB,PLT,CYTOGENETICS
str,str,f64,f64,f64,f64,f64,f64,str
"""P132697""","""MSK""",14.0,2.8,0.2,0.7,7.6,119.0,"""46,xy,del(20)(q12)[2]/46,xy[18…"
"""P132698""","""MSK""",1.0,7.4,2.4,0.1,11.6,42.0,"""46,xx"""
"""P116889""","""MSK""",15.0,3.7,2.1,0.1,14.2,81.0,"""46,xy,t(3;3)(q25;q27)[8]/46,xy…"
"""P132699""","""MSK""",1.0,3.9,1.9,0.1,8.9,77.0,"""46,xy,del(3)(q26q27)[15]/46,xy…"
"""P132700""","""MSK""",6.0,128.0,9.7,0.9,11.1,195.0,"""46,xx,t(3;9)(p13;q22)[10]/46,x…"


In [23]:
df.shape

(3323, 9)

In [24]:
df.describe()

statistic,ID,CENTER,BM_BLAST,WBC,ANC,MONOCYTES,HB,PLT,CYTOGENETICS
str,str,str,f64,f64,f64,f64,f64,f64,str
"""count""","""3323""","""3323""",3214.0,3051.0,3130.0,2722.0,3213.0,3199.0,"""2936"""
"""null_count""","""0""","""0""",109.0,272.0,193.0,601.0,110.0,124.0,"""387"""
"""mean""",null,null,5.982545,6.535164,3.264735,0.955868,9.893549,167.0489,null
"""std""",null,null,7.615439,10.247219,5.237043,2.666478,2.041158,149.477031,null
"""min""","""P100000""","""CCH""",0.0,0.2,0.0,0.0,4.0,2.0,"""+8(fish)"""
"""25%""",null,null,1.0,2.7,1.0,0.15,8.5,66.0,null
"""50%""",null,null,3.0,4.1,2.0,0.37,9.7,123.0,null
"""75%""",null,null,8.0,6.66,3.69,0.784,11.2,230.0,null
"""max""","""P132729""","""VU""",91.0,154.4,109.62,44.2,16.6,1451.0,"""tris8"""


In [ ]:
df["CENTER"].value_counts() # The clinical center where the patient is treated.

CENTER,count
str,u32
"""CGM""",107
"""TUD""",73
"""MUV""",83
"""UOB""",88
"""IHBT""",33
…,…
"""FLO""",68
"""DUTH""",66
"""PV""",316


In [ ]:
mol_df.head() # Molecular data set

ID,CHR,START,END,REF,ALT,GENE,PROTEIN_CHANGE,EFFECT,VAF,DEPTH
str,str,f64,f64,str,str,str,str,str,f64,f64
"""P100000""","""11""",1.19149248e8,1.19149248e8,"""G""","""A""","""CBL""","""p.C419Y""","""non_synonymous_codon""",0.083,1308.0
"""P100000""","""5""",1.31822301e8,1.31822301e8,"""G""","""T""","""IRF1""","""p.Y164*""","""stop_gained""",0.022,532.0
"""P100000""","""3""",7.769406e7,7.769406e7,"""G""","""C""","""ROBO2""","""p.?""","""splice_site_variant""",0.41,876.0
"""P100000""","""4""",1.06164917e8,1.06164917e8,"""G""","""T""","""TET2""","""p.R1262L""","""non_synonymous_codon""",0.43,826.0
"""P100000""","""2""",2.5468147e7,2.5468163e7,"""ACGAAGAGGGGGTGTTC""","""A""","""DNMT3A""","""p.E505fs*141""","""frameshift_variant""",0.0898,942.0


In [28]:
mol_df.shape

(10935, 11)

In [54]:
# Number of unique patients
mol_df["ID"].unique()

ID
str
"""P102741"""
"""P100248"""
"""P117453"""
"""P110437"""
"""P105559"""
…
"""P100159"""
"""P105962"""
"""P102655"""


In [55]:
mol_df["CHR"].unique() # Number of all possible mutations

CHR
str
"""18"""
"""21"""
"""16"""
"""22"""
"""15"""
…
"""11"""
"""12"""
"""13"""


In [56]:
mol_df.group_by("ID").agg(pl.col("CHR").count()) # Distribution of the number of mutations per patient

ID,CHR
str,u32
"""P120965""",8
"""P100265""",7
"""P100027""",4
"""P110856""",9
"""P100055""",5
…,…
"""P120833""",6
"""P102858""",3
"""P102981""",1


In [ ]:
mol_df["GENE"].value_counts().sort(by="count" , descending=True).head(5) # The 5 most affected genes 

GENE,count
str,u32
"""TET2""",1663
"""ASXL1""",951
"""SF3B1""",775
"""DNMT3A""",604
"""RUNX1""",578


In [49]:
target_df.head()

ID,OS_YEARS,OS_STATUS
str,f64,f64
"""P132697""",1.115068,1.0
"""P132698""",4.928767,0.0
"""P116889""",2.043836,0.0
"""P132699""",2.476712,1.0
"""P132700""",3.145205,0.0


In [51]:
target_df["OS_STATUS"].value_counts()

OS_STATUS,count
f64,u32
1.0,1600
null,150
0.0,1573


In [53]:
target_df.describe()

statistic,ID,OS_YEARS,OS_STATUS
str,str,f64,f64
"""count""","""3323""",3173.0,3173.0
"""null_count""","""0""",150.0,150.0
"""mean""",null,2.480713,0.504255
"""std""",null,2.588259,0.500061
"""min""","""P100000""",0.0,0.0
"""25%""",null,0.652055,0.0
"""50%""",null,1.652055,1.0
"""75%""",null,3.572603,1.0
"""max""","""P132729""",22.043836,1.0
